# Backtesting basics: violations + Kupiec POF (LR test)

**Updated:** 2026-03-05  
**Goal:** build an end-to-end backtesting pipeline (rolling VaR → violations → Kupiec POF p-value) on sample PnL data.

## What we test
- **Violation** occurs when realized loss exceeds the rolling VaR threshold: 

  $$ I_t = \mathbf{1}_{\{L_t > \mathrm{VaR}_{\alpha,t}\}} $$

- **Kupiec POF** tests whether the observed violation rate matches the expected rate $1-\alpha$.

## 1) Load sample PnL and define loss
We use the project convention `loss = -pnl`.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/sample_pnl.csv")
df["date"] = pd.date_range("2026-01-01", periods=len(df), freq="D")
df = df.set_index("date").sort_index()

pnl = df["pnl"]
loss = -pnl

df.head()


## 2) Rolling VaR, violations, and Kupiec POF test
Compute rolling historical VaR and define violations as `loss > VaR`. Then test whether the observed violation rate is consistent with the expected rate `1 - alpha`.

In [ ]:
from riskmetrics.var import rolling_historical_var
from riskmetrics.backtest import var_violations, kupiec_pof_test

alpha = 0.99
window = 3  # demo window; realistic is ~250 for daily trading

rvar = rolling_historical_var(pnl, window=window, alpha=alpha)
viol = var_violations(loss, rvar)
out = kupiec_pof_test(viol, alpha=alpha)

rvar.tail(8), viol.tail(8), out


## 3) Sanity check
Verify that the violation flags are correct by inspecting `loss` and `VaR`.

In [ ]:
aligned = pd.concat([loss.rename("loss"), rvar.rename("VaR")], axis=1).dropna()
aligned["violation"] = (aligned["loss"] > aligned["VaR"]).astype(int)
aligned

## 4) Summary
- With `alpha=0.99` and `window=3`, we observe a high violation rate on this tiny sample (pipeline sanity check).
- Kupiec POF returns LR and a p-value for coverage testing.
- For meaningful backtesting, we need a longer time series (e.g., >= 250 observations) and realistic windows.

---

## 5) Next
- Run the same pipeline on a longer PnL series.
- Add a compact backtest report (violation rate, Kupiec p-value) and a plot (loss vs rolling VaR with violation markers).


---



# Backtesting: rolling VaR coverage on a long series (Kupiec POF + diagnostics)

**Updated:** 2026-03-06  
**Goal:** run the backtesting pipeline on a longer series (>=250) and produce a compact report + diagnostic plot.

## 1) Load long sample PnL series

We generate a long daily PnL series using random samples from a Normal distribution, $\mathrm{PnL}_t \sim \mathcal{N}(0, 0.01^2)$, i.e., mean 0 and 1% daily volatility.

We also create a synthetic daily date index to make time-series operations and plots readable.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/long_pnl.csv")
df["date"] = pd.date_range("2023-01-01", periods=len(df), freq="D")
df = df.set_index("date").sort_index()

pnl = df["pnl"]
loss = -pnl
df.head()

## 2) Rolling VaR, violations, and Kupiec POF test

**Step A:** Convert PnL to loss using the project convention `loss = -pnl`.  
**Step B:** Compute rolling historical VaR with a realistic window (e.g., 250 trading days).  
**Step C:** Define violations as `loss > VaR`.  
**Step D:** Run Kupiec’s POF test to check whether the observed violation rate matches the expected rate \(1-\alpha\).

Notes:
- The first `window - 1` rows are `NaN` for rolling VaR because the window is not yet full.
- Index alignment matters: we compute violations after aligning `loss` and `VaR` on the same dates.

In [ ]:
from riskmetrics.var import rolling_historical_var
from riskmetrics.backtest import var_violations, kupiec_pof_test

alpha = 0.99
window = 250

rvar = rolling_historical_var(pnl, window=window, alpha=alpha)
viol = var_violations(loss, rvar)
out = kupiec_pof_test(viol, alpha=alpha)

out

## 3) Compact backtest report

To make the backtest easy to reuse, we summarize key results into a compact report:

- `n`: number of aligned observations used in backtesting
- `x`: number of violations
- `observed_rate`: $ x/n $
- `expected_rate`: $ 1-\alpha $
- `lr_pof` and `p_value`: Kupiec POF likelihood ratio statistic and its chi-square p-value

In [ ]:
from riskmetrics.backtest import backtest_report
rep = backtest_report(loss, rvar, alpha=alpha)
rep

## 4) Diagnostic table (alignment check)

As a sanity check, we build a table containing `loss`, `VaR`, and the `violation` flag on the same index.  
This makes it easy to confirm that violations occur exactly when `loss > VaR`.

In [ ]:
aligned = pd.concat([loss.rename("loss"), rvar.rename("VaR")], axis=1).dropna()
aligned["violation"] = (aligned["loss"] > aligned["VaR"]).astype(int)
aligned.tail(10)

## 5) Diagnostic plot

We plot the loss series and the rolling VaR threshold, and mark violation points.  
This provides a quick visual validation of alignment and threshold breaches.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(aligned.index, aligned["loss"], label="loss")
plt.plot(aligned.index, aligned["VaR"], label="rolling VaR")

v = aligned[aligned["violation"] == 1]
plt.scatter(v.index, v["loss"], marker="o", label="violation")

plt.title(f"Loss vs rolling VaR (alpha={alpha}, window={window})")
plt.xlabel("date")
plt.ylabel("value")
plt.legend()
plt.tight_layout()
plt.show()

## 6) Summary
- Ran rolling historical VaR backtesting on a long series (n>=250).
- Produced a compact report (n, x, observed vs expected rate, LR_POF, p-value).
- Visual check confirms violations occur when loss exceeds the rolling VaR threshold.

---